<a href="https://colab.research.google.com/github/WinsalotNot/HAHA_v1/blob/master/Assignment_2_Multiclass_Perceptron_Andrew.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **IMPORTS**

In [2]:
import numpy as np
import pandas as pd

# **DATA ORGANIZATION**

In [11]:
# Stores data in data_file (does not include header!)
data_file = pd.read_csv('https://gist.githubusercontent.com/netj/8836201/raw/6f9306ad21398ea43cba4f7d537619d0e07d5ae3/iris.csv')
print(data_file)

# Get the unqiue values in variety for classification
unique_variety = data_file['variety'].unique()
print(unique_variety)

# Take 80% of the dataset RANDOMLY as training data, random_state is the random seed used to ensure reproducibility
training_data_80 = data_file.sample(frac=0.8, random_state=25)
# Take 20% of the dataset by ELIMINATING the training data
testing_data_20 = data_file.drop(training_data_80.index)
print(f'Training Data 1: {(len(training_data_80)/len(data_file)*100)}% Testing Data 1: {(len(testing_data_20)/len(data_file)*100)}%')

# Take 70% of the dataset RANDOMLY as training data, random_state is the random seed used to ensure reproducibility
training_data_70 = data_file.sample(frac=0.7, random_state=24)
# Take 30% of the dataset by ELIMINATING the training data
testing_data_30 = data_file.drop(training_data_70.index)
print(f'Training Data 2: {(len(training_data_70)/len(data_file)*100)}% Testing Data 2: {(len(testing_data_30)/len(data_file)*100)}%')

# Take 60% of the dataset RANDOMLY as training data, random_state is the random seed used to ensure reproducibility
training_data_60 = data_file.sample(frac=0.6, random_state=23)
# Take 40% of the dataset by ELIMINATING the training data
testing_data_40 = data_file.drop(training_data_60.index)
print(f'Training Data 3: {(len(training_data_60)/len(data_file)*100)}% Testing Data 3: {(len(testing_data_40)/len(data_file)*100)}%')


     sepal.length  sepal.width  petal.length  petal.width    variety
0             5.1          3.5           1.4          0.2     Setosa
1             4.9          3.0           1.4          0.2     Setosa
2             4.7          3.2           1.3          0.2     Setosa
3             4.6          3.1           1.5          0.2     Setosa
4             5.0          3.6           1.4          0.2     Setosa
..            ...          ...           ...          ...        ...
145           6.7          3.0           5.2          2.3  Virginica
146           6.3          2.5           5.0          1.9  Virginica
147           6.5          3.0           5.2          2.0  Virginica
148           6.2          3.4           5.4          2.3  Virginica
149           5.9          3.0           5.1          1.8  Virginica

[150 rows x 5 columns]
['Setosa' 'Versicolor' 'Virginica']
Training Data 1: 80.0% Testing Data 1: 20.0%
Training Data 2: 70.0% Testing Data 2: 30.0%
Training Data 3: 60.0%

# **MULTICLASS PERCEPTRON ALGORITHM**

In [30]:
def multiclass_perceptron(data, weight_2D_array, learning_rate, epoch, isTesting):
  # Takes all row and all except last [:, :-1], then converts to numpy compatible array
  data_numpy = data.iloc[:, :-1].to_numpy()

  # Each unqiue data in 'variety' are represented sequentially from 0
  result_each_indexes = data.iloc[:, -1].to_numpy()                                         # Takes all rows and ONLY the last column [:, -1], then converts to numpy compatible array
  unique_labels, results_given_int = np.unique(result_each_indexes, return_inverse=True)    # Gets an array of the labels of each uniques and an array of the unique values mapped to their respective indexes
  label_to_index = {label: idx for idx, label in enumerate(unique_labels)}                  # Shows which unqiue values are assigned to which index
  print("Label Mapping:", label_to_index)
  print("Converted Indexes:", results_given_int)

  updated_weights = weight_2D_array.copy()
  best_weights = updated_weights.copy()
  best_accuracy = 0.0

  if isTesting:
    weighted_sums_testing = np.dot(data_numpy, updated_weights.T)
    results_calculated_testing = np.argmax(weighted_sums_testing, axis=1)
    correct_predictions_testing = np.sum(results_calculated_testing == results_given_int)
    accuracy_testing = (correct_predictions_testing / len(data_numpy) * 100)

    print(f'Testing Results: {results_calculated_testing}')
    print(f'Compared To: {results_given_int}')

    return accuracy_testing


  else:
    iteration = 0
    success = False
    while (iteration < epoch):
      weighted_sums = np.dot(data_numpy, updated_weights.T) # Uses dot matrix calculation as well as tansposing the updated_weights (turning the features into rows and classes in columns)
      results_calculated = np.argmax(weighted_sums, axis=1) # Because classes are now rows, take the biggest one from each row

      correct_predictions = np.sum(results_calculated == results_given_int)
      accuracy = correct_predictions / len(data_numpy)  # Compute accuracy

      # Track best accuracy and corresponding weights
      if accuracy > best_accuracy:
          best_accuracy = accuracy
          best_weights = updated_weights.copy()

      print(f'Epoch {iteration + 1}: {correct_predictions}/{len(data_numpy)} samples correctly classified.')

      if correct_predictions == len(data_numpy):  # If 100% correct, terminate early
          print(f'Converged at epoch {iteration + 1}')
          success = True
          break

      for index, (result_calculated, result_given_int) in enumerate(zip(results_calculated, results_given_int)):
        if result_calculated != result_given_int:
            updated_weights[result_given_int] += learning_rate * data_numpy[index]  # Increases the expected class
            updated_weights[result_calculated] -= learning_rate * data_numpy[index] # Decreases the predicted class

      iteration += 1

    if not success:
      print(f'Given {epoch} epoch, weights did not converge: {updated_weights}')

    print(f'Best accuracy: {best_accuracy * 100:.2f}%')
    print(f'Best weights:\n{best_weights}')
    return best_weights, (best_accuracy * 100)


In [43]:
weight_2D_array_zeros = np.array([
    [0., 0., 0., 0.],
    [0., 0., 0., 0.],
    [0., 0., 0., 0.]
])
# After Transposing:
# [
#   [0, 0, 0],
#   [0, 0, 0],
#   [0, 0, 0],
#   [0, 0, 0]
# ]

# Randomly create a 2D array with 3
np.random.seed(26)
rows, column = 4, 3
weight_2D_array_range = np.random.uniform(-0.5, 0.5, (column, rows))
print(weight_2D_array_range)

# Case: 80/20 Split, Zeros Weight, 0.1 Learning Rate
amazing_weights_8020_0_0dot1, accuracy_training_8020_0_0dot1 = multiclass_perceptron(training_data_80, weight_2D_array_zeros, 0.1, 1000, False)
accuracy_testing_8020_0_0dot1 = multiclass_perceptron(testing_data_20, amazing_weights_8020_0_0dot1, None, None, True)

# Case: 80/20 Split, Range Weight, 0.1 Learning Rate
amazing_weights_8020_R_0dot1, accuracy_training_8020_R_0dot1 = multiclass_perceptron(training_data_80, weight_2D_array_range, 0.1, 1000, False)
accuracy_testing_8020_R_0dot1 = multiclass_perceptron(testing_data_20, amazing_weights_8020_R_0dot1, None, None, True)

# Case: 80/20 Split, Zero Weight, 0.01 Learning Rate
amazing_weights_8020_0_0dot01, accuracy_training_8020_0_0dot01 = multiclass_perceptron(training_data_80, weight_2D_array_zeros, 0.01, 1000, False)
accuracy_testing_8020_0_0dot01 = multiclass_perceptron(testing_data_20, amazing_weights_8020_0_0dot01, None, None, True)

# Case: 80/20 Split, Range Weight, 0.01 Learning Rate
amazing_weights_8020_R_0dot01, accuracy_training_8020_R_0dot01 = multiclass_perceptron(training_data_80, weight_2D_array_range, 0.01, 1000, False)
accuracy_testing_8020_R_0dot01 = multiclass_perceptron(testing_data_20, amazing_weights_8020_R_0dot01, None, None, True)

# Case: 70/30 Split, Zeros Weight, 0.1 Learning Rate
amazing_weights_7030_0_0dot1, accuracy_training_7030_0_0dot1 = multiclass_perceptron(training_data_70, weight_2D_array_zeros, 0.1, 1000, False)
accuracy_testing_7030_0_0dot1 = multiclass_perceptron(testing_data_30, amazing_weights_7030_0_0dot1, None, None, True)

# Case: 70/30 Split, Range Weight, 0.1 Learning Rate
amazing_weights_7030_R_0dot1, accuracy_training_7030_R_0dot1 = multiclass_perceptron(training_data_70, weight_2D_array_range, 0.1, 1000, False)
accuracy_testing_7030_R_0dot1 = multiclass_perceptron(testing_data_30, amazing_weights_7030_R_0dot1, None, None, True)

# Case: 70/30 Split, Zero Weight, 0.01 Learning Rate
amazing_weights_7030_0_0dot01, accuracy_training_7030_0_0dot01 = multiclass_perceptron(training_data_70, weight_2D_array_zeros, 0.01, 1000, False)
accuracy_testing_7030_0_0dot01 = multiclass_perceptron(testing_data_30, amazing_weights_7030_0_0dot01, None, None, True)

# Case: 70/30 Split, Range Weight, 0.01 Learning Rate
amazing_weights_7030_R_0dot01, accuracy_training_7030_R_0dot01 = multiclass_perceptron(training_data_70, weight_2D_array_range, 0.01, 1000, False)
accuracy_testing_7030_R_0dot01 = multiclass_perceptron(testing_data_30, amazing_weights_7030_R_0dot01, None, None, True)

# Case: 60/40 Split, Zeros Weight, 0.1 Learning Rate
amazing_weights_6040_0_0dot1, accuracy_training_6040_0_0dot1 = multiclass_perceptron(training_data_60, weight_2D_array_zeros, 0.1, 1000, False)
accuracy_testing_6040_0_0dot1 = multiclass_perceptron(testing_data_40, amazing_weights_6040_0_0dot1, None, None, True)

# Case: 60/40 Split, Range Weight, 0.1 Learning Rate
amazing_weights_6040_R_0dot1, accuracy_training_6040_R_0dot1 = multiclass_perceptron(training_data_60, weight_2D_array_range, 0.1, 1000, False)
accuracy_testing_6040_R_0dot1 = multiclass_perceptron(testing_data_40, amazing_weights_6040_R_0dot1, None, None, True)

# Case: 60/40 Split, Zero Weight, 0.01 Learning Rate
amazing_weights_6040_0_0dot01, accuracy_training_6040_0_0dot01 = multiclass_perceptron(training_data_60, weight_2D_array_zeros, 0.01, 1000, False)
accuracy_testing_6040_0_0dot01 = multiclass_perceptron(testing_data_40, amazing_weights_6040_0_0dot01, None, None, True)

# Case: 60/40 Split, Range Weight, 0.01 Learning Rate
amazing_weights_6040_R_0dot01, accuracy_training_6040_R_0dot01 = multiclass_perceptron(training_data_60, weight_2D_array_range, 0.01, 1000, False)
accuracy_testing_6040_R_0dot01 = multiclass_perceptron(testing_data_40, amazing_weights_6040_R_0dot01, None, None, True)

print(f'\n\nCase: 80/20 Split, Zeros Weight, 0.1 Learning Rate')
print(f'--------------------------------------------------')
print(f'Training Accuracy: {accuracy_training_8020_0_0dot1}%')
print(f'Testing Accuracy: {accuracy_testing_8020_0_0dot1}%')
print(f'\nCase: 80/20 Split, Range Weight, 0.1 Learning Rate')
print(f'--------------------------------------------------')
print(f'Training Accuracy: {accuracy_training_8020_R_0dot1}%')
print(f'Testing Accuracy: {accuracy_testing_8020_R_0dot1}%')
print(f'\nCase: 80/20 Split, Zero Weight, 0.01 Learning Rate')
print(f'--------------------------------------------------')
print(f'Training Accuracy: {accuracy_training_8020_0_0dot01}%')
print(f'Testing Accuracy: {accuracy_testing_8020_0_0dot01}%')
print(f'\nCase: 80/20 Split, Range Weight, 0.01 Learning Rate')
print(f'---------------------------------------------------')
print(f'Training Accuracy: {accuracy_training_8020_R_0dot01}%')
print(f'Testing Accuracy: {accuracy_testing_8020_R_0dot01}%')
print(f'\nCase: 70/30 Split, Zeros Weight, 0.1 Learning Rate')
print(f'--------------------------------------------------')
print(f'Training Accuracy: {accuracy_training_7030_0_0dot1}%')
print(f'Testing Accuracy: {accuracy_testing_7030_0_0dot1}%')
print(f'\nCase: 70/30 Split, Range Weight, 0.1 Learning Rate')
print(f'--------------------------------------------------')
print(f'Training Accuracy: {accuracy_training_7030_R_0dot1}%')
print(f'Testing Accuracy: {accuracy_testing_7030_R_0dot1}%')
print(f'\nCase: 70/30 Split, Zero Weight, 0.01 Learning Rate')
print(f'--------------------------------------------------')
print(f'Training Accuracy: {accuracy_training_7030_0_0dot01}%')
print(f'Testing Accuracy: {accuracy_testing_7030_0_0dot01}%')
print(f'\nCase: 70/30 Split, Range Weight, 0.01 Learning Rate')
print(f'---------------------------------------------------')
print(f'Training Accuracy: {accuracy_training_7030_R_0dot01}%')
print(f'Testing Accuracy: {accuracy_testing_7030_R_0dot01}%')
print(f'\nCase: 60/40 Split, Zeros Weight, 0.1 Learning Rate')
print(f'--------------------------------------------------')
print(f'Training Accuracy: {accuracy_training_6040_0_0dot1}%')
print(f'Testing Accuracy: {accuracy_testing_6040_0_0dot1}%')
print(f'\nCase: 60/40 Split, Range Weight, 0.1 Learning Rate')
print(f'--------------------------------------------------')
print(f'Training Accuracy: {accuracy_training_6040_R_0dot1}%')
print(f'Testing Accuracy: {accuracy_testing_6040_R_0dot1}%')
print(f'\nCase: 60/40 Split, Zero Weight, 0.01 Learning Rate')
print(f'--------------------------------------------------')
print(f'Training Accuracy: {accuracy_training_6040_0_0dot01}%')
print(f'Testing Accuracy: {accuracy_testing_6040_0_0dot01}%')
print(f'\nCase: 60/40 Split, Range Weight, 0.01 Learning Rate')
print(f'---------------------------------------------------')
print(f'Training Accuracy: {accuracy_training_6040_R_0dot01}%')
print(f'Testing Accuracy: {accuracy_testing_6040_R_0dot01}%')

Streaming output truncated to the last 5000 lines.
Epoch 153: 69/105 samples correctly classified.
Epoch 154: 42/105 samples correctly classified.
Epoch 155: 68/105 samples correctly classified.
Epoch 156: 94/105 samples correctly classified.
Epoch 157: 93/105 samples correctly classified.
Epoch 158: 87/105 samples correctly classified.
Epoch 159: 83/105 samples correctly classified.
Epoch 160: 68/105 samples correctly classified.
Epoch 161: 69/105 samples correctly classified.
Epoch 162: 68/105 samples correctly classified.
Epoch 163: 73/105 samples correctly classified.
Epoch 164: 68/105 samples correctly classified.
Epoch 165: 72/105 samples correctly classified.
Epoch 166: 68/105 samples correctly classified.
Epoch 167: 77/105 samples correctly classified.
Epoch 168: 68/105 samples correctly classified.
Epoch 169: 59/105 samples correctly classified.
Epoch 170: 68/105 samples correctly classified.
Epoch 171: 84/105 samples correctly classified.
Epoch 172: 68/105 samples correctly c